# Orchestrator baselines · Kev-4B (zero-shot) on Kaggle

Plan: `plans/P1-kev-0.8b.md (pattern; 4B has no separate plan)`. Stages run in order and the notebook **stops at the first failure**:
`smoke` (5 rows, one per route, asserts) → `validation` (298) → `test` (593) → report.

**Before you run**
- Settings → Accelerator: **GPU T4 x2** (one GPU is used). Settings → Internet: **On** (for PyPI and Hugging Face).
- Add Input → your private Dataset made from `dist/kev-bundle/` (`python kev/make_bundle.py`). No git clone of the project repo is needed.
- Optional secret `HF_TOKEN`.
- Smoke only: set `STAGES = ["smoke"]`. Everything: leave as is, then *Save Version → Save & Run All (Commit)*.

Results are written to `/kaggle/working/runs/kev/<RUN_ID>/` and are saved as the notebook output.
Heavy things (venv, upstream Kev clone, weights) live in `/tmp` so they are not saved with the output.


In [ ]:
# ---- CONFIG (edit here) ----
MODEL = "jaredpalmer/kev-4b@139fdd94f1b6a6ad80cc15e08fcb99cac885a101"   # weights commit named by the model card (tags are movable)
KEV_REPO = "https://github.com/jaredpalmer/kev.git"   # upstream, public
KEV_SHA = "2074cc44256200bb37540103fa4572901faa10f3"   # commit the adapter was written against
FLA_VERSION = "0.5.2"                    # flash-linear-attention pinned by kev/fused_qwen35.py
STAGES = ["smoke", "validation", "test"]

import datetime, os
RUN_ID = "kev-4b-" + datetime.date.today().strftime("%Y%m%d") + "-a"
OUT = "/kaggle/working/runs/kev"
WORK = "/tmp/work"
os.makedirs(WORK, exist_ok=True)
os.environ["HF_HOME"] = "/tmp/hf"
print(RUN_ID, STAGES)


In [ ]:
import subprocess, sys, shlex

def sh(cmd, cwd=None, check=True, env=None):
    """Run a command, stream its output into the cell, raise on failure."""
    print("$", cmd if isinstance(cmd, str) else shlex.join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv")
sh("python --version; df -h /tmp /kaggle/working | cat; free -g | head -2")

In [ ]:
# ---- 1. code: project bundle (attached Dataset) + upstream Kev at the pinned commit ----
import glob
try:
    from kaggle_secrets import UserSecretsClient
    try: os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
except ImportError:
    pass   # not on Kaggle

hits = glob.glob("/kaggle/input/**/orchestrator-baselines/common/data.py", recursive=True)
assert len(hits) == 1, f"attach exactly one Dataset built by kev/make_bundle.py; found {hits}"
BASE = os.path.dirname(os.path.dirname(hits[0]))          # .../orchestrator-baselines (read-only)
assert os.path.isdir(os.path.join(os.path.dirname(BASE), "data-generation", "data", "final", "test")), "data folder missing from the Dataset"
print("project bundle:", BASE)

KEV_DIR = f"{WORK}/kev-src"
if not os.path.exists(KEV_DIR):
    sh(f"git clone {KEV_REPO} {KEV_DIR}")
sh(f"git checkout -q {KEV_SHA} && git rev-parse HEAD", cwd=KEV_DIR)


In [ ]:
# ---- 2. environment: own venv (Python 3.12, kev's locked deps) ----
sh("pip -q install uv")
sh("uv sync --extra serve --no-dev", cwd=KEV_DIR, env={"UV_PROJECT_ENVIRONMENT": f"{WORK}/venv"})
PY = f"{WORK}/venv/bin/python"
sh(f"uv pip install --python {PY} flash-linear-attention=={FLA_VERSION}")
sh(f"uv pip install --python {PY} -r {BASE}/requirements-common.txt")
sh([PY, "-c", "import torch, transformers, fla; print('torch', torch.__version__, '| transformers', transformers.__version__, '| fla', fla.__version__)"])

In [ ]:
# ---- 3. GPU sanity (fails early with a clear message) ----
sh([PY, "-c", '''
import torch
assert torch.cuda.is_available(), "no CUDA GPU: set Accelerator to GPU T4 x2"
cap = torch.cuda.get_device_capability(0)
arch = f"sm_{cap[0]}{cap[1]}"
print("device", torch.cuda.get_device_name(0), arch, "| bf16 supported:", torch.cuda.is_bf16_supported(), "| arch list:", torch.cuda.get_arch_list())
assert arch in torch.cuda.get_arch_list(), f"this torch build has no kernels for {arch} (a P100 is sm_60): choose a T4"
x = torch.randn(256, 256, device="cuda"); print("matmul ok", float((x @ x).sum()) == float((x @ x).sum()))
'''])
# shared package tests (data audit invariants + dispatcher oracle) on this machine
sh(f"{PY} -m pytest -q -p no:cacheprovider common/tests kev/tests", cwd=BASE, env={"PYTHONPATH": KEV_DIR})

In [ ]:
# ---- 4. SMOKE TEST: 5 validation rows, one per gold route ----
import json, pathlib

def run_kev(split, extra=""):
    return sh(f"{PY} kev/run.py --split {split} --run-id {RUN_ID} --out {OUT} --model {MODEL} --dtype bf16 "
              f"--kev-src {KEV_DIR} --platform kaggle {extra}", cwd=BASE)

if "smoke" in STAGES:
    run_kev("validation", "--smoke")                       # exit code != 0 on any error row or validator problem
    sdir = pathlib.Path(OUT) / f"{RUN_ID}-smoke"
    recs = [json.loads(l) for l in open(sdir / "validation.jsonl")]
    assert len(recs) == 5, len(recs)
    assert all(r["error"] is None for r in recs)
    assert all(abs(sum(r["answers"][k]["probs"].values()) - 1) < 1e-3 for r in recs for k in ("scope_check", "feature_classification", "complexity_estimation"))
    run = json.load(open(sdir / "run.json"))
    print("\ndevice", run["device"], "| dtype", run["dtype"], "| temperature", run["temperature_applied"],
          "| revision", run["model_revision"], "| params", run["model_size"]["params_total"],
          "| peak GPU MB", run["model_size"]["peak_gpu_mem_mb"], "| load s", run["load_time_s"])
    for r in recs:
        a = r["answers"]
        print(r["query_id"], "| scope", a["scope_check"]["label"], "| feature", a["feature_classification"]["label"],
              "| slots", {k: v["label"] for k, v in a["slots"].items()}, "| %.0f ms" % r["latency_ms"],
              "| tok native/ref", r["tokens"]["input_native"], r["tokens"]["input_ref"])
    # P1 step 4: what does the token count mean (1 vs 2 vs 4 questions)
    sh(f"{PY} kev/tokens_check.py --model {MODEL} --dtype bf16 --out {sdir}/tokens_check.json", cwd=BASE)
    print("\nSMOKE TEST PASSED")
else:
    print("smoke skipped")

In [ ]:
# ---- 5. full runs (resumable: re-running skips query_ids already in the log) ----
for split in ("validation", "test"):
    if split in STAGES:
        run_kev(split)

In [ ]:
# ---- 6. shared report (only if full logs exist) ----
run_dir = f"{OUT}/{RUN_ID}"
if os.path.exists(run_dir):
    sh(f"{PY} kev/make_report.py {run_dir}", cwd=BASE)
    for split in ("validation", "test"):
        p = pathlib.Path(run_dir) / f"report_{split}.md"
        if p.exists():
            print("\n" + "=" * 20, p.name); print(p.read_text()[:6000])
sh(f"ls -la {OUT}/* | head -40; du -sh /kaggle/working")